In [15]:
import ConnectionConfig as cc
from pyspark.sql.functions import *
cc.setupEnvironment()
cc.listEnvironment()

ALLUSERSPROFILE: C:\ProgramData
APPDATA: C:\Users\lzkol\AppData\Roaming
COMMONPROGRAMFILES: C:\Program Files\Common Files
COMMONPROGRAMFILES(X86): C:\Program Files (x86)\Common Files
COMMONPROGRAMW6432: C:\Program Files\Common Files
COMPUTERNAME: LIZA
COMSPEC: C:\WINDOWS\system32\cmd.exe
DRIVERDATA: C:\Windows\System32\Drivers\DriverData
EFC_13296_1262719628: 1
EFC_13296_1592913036: 1
EFC_13296_2283032206: 1
EFC_13296_2775293581: 1
EFC_13296_3789132940: 1
FPS_BROWSER_APP_PROFILE_STRING: Internet Explorer
FPS_BROWSER_USER_PROFILE_STRING: Default
GOPATH: C:\Users\lzkol\go
HOMEDRIVE: C:
HOMEPATH: \Users\lzkol
IPY_INTERRUPT_EVENT: 1964
JAVA_HOME: C:\Program Files\Java\jdk-11\
JPY_INTERRUPT_EVENT: 1964
JPY_PARENT_PID: 2020
JPY_SESSION_NAME: S1_03_DIM_WEATHER.ipynb
LANG: en_US.UTF-8
LANGUAGE: 
LC_ALL: en_US.UTF-8
LOCALAPPDATA: C:\Users\lzkol\AppData\Local
LOGONSERVER: \\LIZA
NUMBER_OF_PROCESSORS: 16
ONEDRIVE: C:\Users\lzkol\OneDrive
OS: Windows_NT
PATH: C:\Users\lzkol\PycharmProjects\sparkde

In [16]:
spark = cc.startLocalCluster("dimWeather")
spark.getActiveSession()

In [19]:
# EXTRACT
df_weather_dim = spark.read.option("header", True).csv("weather_dim.csv")

# TRANSFORM
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit

window_spec = Window.orderBy(lit(1))

df_weather_dim = df_weather_dim.withColumn("weather_id", row_number().over(window_spec))

# Inspect schema and preview
df_weather_dim.printSchema()
df_weather_dim.show()

# LOAD
df_weather_dim.write.format("delta").mode("overwrite").saveAsTable("dimWeather")


root
 |-- weather_type: string (nullable = true)
 |-- weather_id: integer (nullable = false)

+------------+----------+
|weather_type|weather_id|
+------------+----------+
|    Pleasant|         1|
|  Unpleasant|         2|
|     Neutral|         3|
|     Unknown|         4|
+------------+----------+



In [7]:
print(cc.create_jdbc())

jdbc:postgresql://localhost:5432/tutorial_op?user=postgres&password=Student_1234&ssl=false


## Delete the spark session

In [13]:
import requests
import json
import os

# Postal codes and their coordinates
postal_codes = {
    "2050": (51.2199, 4.4035),
    "2140": (51.2096, 4.4354),
    "2660": (51.1902, 4.4326),
    "2100": (51.1474, 4.3013),
    "2000": (51.2199, 4.4035),
    "2600": (51.1902, 4.4326),
    "2018": (51.2199, 4.4035),
    "2020": (51.2199, 4.4035),
    "2610": (51.1673, 4.3951),
    "2170": (51.2462, 4.449),
    "2030": (51.2199, 4.4035),
    "2060": (51.2199, 4.4035)
}

dates = ["2023-06-01", "2023-06-02", "2023-06-03"]

output_folder = r"C:\Users\lzkol\PycharmProjects\sparkdelta\weather"
os.makedirs(output_folder, exist_ok=True)

for zip_code, (lat, lon) in postal_codes.items():
    for i, date in enumerate(dates):
        url = (
            f"https://archive-api.open-meteo.com/v1/archive?"
            f"latitude={lat}&longitude={lon}"
            f"&start_date={date}&end_date={date}"
            f"&hourly=temperature_2m,weathercode"
            f"&timezone=Europe/Brussels"
        )
        response = requests.get(url)
        data = response.json()
        data["zipCode"] = zip_code
        data["date"] = date
        with open(f"{output_folder}/weather_{zip_code}_{date}.json", "w") as f:
            json.dump(data, f, indent=2)
        print(f"Saved weather for {zip_code} on {date}")

Saved weather for 2050 on 2023-06-01
Saved weather for 2050 on 2023-06-02
Saved weather for 2050 on 2023-06-03
Saved weather for 2140 on 2023-06-01
Saved weather for 2140 on 2023-06-02
Saved weather for 2140 on 2023-06-03
Saved weather for 2660 on 2023-06-01
Saved weather for 2660 on 2023-06-02
Saved weather for 2660 on 2023-06-03
Saved weather for 2100 on 2023-06-01
Saved weather for 2100 on 2023-06-02
Saved weather for 2100 on 2023-06-03
Saved weather for 2000 on 2023-06-01
Saved weather for 2000 on 2023-06-02
Saved weather for 2000 on 2023-06-03
Saved weather for 2600 on 2023-06-01
Saved weather for 2600 on 2023-06-02
Saved weather for 2600 on 2023-06-03
Saved weather for 2018 on 2023-06-01
Saved weather for 2018 on 2023-06-02
Saved weather for 2018 on 2023-06-03
Saved weather for 2020 on 2023-06-01
Saved weather for 2020 on 2023-06-02
Saved weather for 2020 on 2023-06-03
Saved weather for 2610 on 2023-06-01
Saved weather for 2610 on 2023-06-02
Saved weather for 2610 on 2023-06-03
S

Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2050_2021-01-01_2021-01-02.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2050_2023-06-01_2023-06-02.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2050_2019-09-22_2019-09-22.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2140_2021-01-01_2021-01-02.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2140_2023-06-01_2023-06-02.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2140_2019-09-22_2019-09-22.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2660_2021-01-01_2021-01-02.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2660_2023-06-01_2023-06-02.json
Data saved to C:\Users\lzkol\PycharmProjects\sparkdelta\weather\weather_data_2660_2019-09-22_2019-09-22.json
Data saved to C:\Us